# Albert Heijn Recipe Scraper + Proxy Pantry Builder

Dit notebook bevat alle stappen voor het scrapen van recepten van Albert Heijn Allerhande en het aanmaken van pantry-proxies voor 100 gebruikers.

**Workflow:**
1. Installeer dependencies
2. Verzamel recept-URLs via de AH API
3. Scrape receptdata (naam, ingrediënten, stappen)
4. Exporteer naar Excel
5. Genereer pantry-proxies voor 100 gebruikers
6. Exporteer pantry-data naar Excel

---
## Stap 0 — Dependencies installeren

In [1]:
!pip install requests beautifulsoup4 openpyxl pandas --quiet
print("Alle dependencies geïnstalleerd")

Alle dependencies geïnstalleerd


---
## Imports

In [2]:
import requests
import time
import re
import random
import pandas as pd
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

random.seed(42)
print("Imports geslaagd")

Imports geslaagd


---
## Stap 1: Configuratie

Pas hier de instellingen aan naar wens.

| Parameter | Beschrijving |
|---|---|
| `AANTAL_RECEPTEN` | Aantal te scrapen recepten (2000 duurt lang, begin met 50–200) |
| `PAGINA_GROOTTE` | Aantal resultaten per API-pagina |
| `DELAY_SECONDEN` | Wachttijd tussen requests (voorkomt server-overload) |
| `URLS_BESTAND` | Bestandsnaam voor opgeslagen URLs |
| `RECEPTEN_BESTAND` | Bestandsnaam voor het Excel-uitvoerbestand |

In [3]:
# ── Instellingen ───────────────────────────────────────────────────────────────
AANTAL_RECEPTEN  = 20        # Verhoog naar 2000 voor volledige dataset
PAGINA_GROOTTE   = 20
DELAY_SECONDEN   = 0.5
URLS_BESTAND     = "recepten_urls.txt"
RECEPTEN_BESTAND = "recepten.xlsx"  #recepten_AH.xlsx is de uiteindelijke dataset, ivm lange runtime wordt in dit bestand een kortere versie gebruikt

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "nl-NL,nl;q=0.9",
}

print(f"Configuratie geladen — {AANTAL_RECEPTEN} recepten worden opgehaald")

Configuratie geladen — 20 recepten worden opgehaald


---
## Stap 2: Authenticatie & URL-verzameling

De AH API vereist een toegangstoken, ook voor anonieme toegang. Dit token wordt 
eenmalig opgehaald en meegegeven aan alle volgende requests. Vervolgens worden 
recept-URLs verzameld door de zoekpagina's van de API te doorlopen. Per pagina 
worden de URLs opgeslagen totdat het gewenste aantal recepten bereikt is.

In [4]:
def haal_token_op() -> str:
    """Haalt een anoniem toegangstoken op van de AH API."""
    resp = requests.post(
        "https://api.ah.nl/mobile-auth/v1/auth/token/anonymous",
        json={"clientId": "appie"},
        headers={"Content-Type": "application/json"},
        timeout=10,
    )
    resp.raise_for_status()
    print("Token opgehaald")
    return resp.json().get("access_token", "")


def haal_urls_op(aantal: int) -> list:
    """Verzamelt recept-URLs via de AH zoek-API."""
    token = haal_token_op()
    urls = []
    pagina = 0

    while len(urls) < aantal:
        resp = requests.get(
            "https://api.ah.nl/mobile-services/recipes/v2/search",
            params={"page": pagina, "size": PAGINA_GROOTTE, "sort": "RELEVANCE"},
            headers={
                "Authorization": f"Bearer {token}",
                "X-Application-Name": "allerhande",
                "User-Agent": "Appie/8.22.3",
            },
            timeout=10,
        )
        resp.raise_for_status()
        content = resp.json().get("content", [])

        if not content:
            print("  Geen resultaten meer — stoppen.")
            break

        for r in content:
            if len(urls) >= aantal:
                break
            recept_id = r.get("id", "")
            slug      = r.get("slugifiedTitle", "")
            urls.append(f"https://www.ah.nl/allerhande/recept/R-R{recept_id}/{slug}")

        print(f"  Pagina {pagina + 1}: {len(urls)}/{aantal} URLs verzameld")
        pagina += 1
        time.sleep(DELAY_SECONDEN)

    return urls

In [5]:
# Verzamel URLs en sla op
urls = haal_urls_op(AANTAL_RECEPTEN)

with open(URLS_BESTAND, "w", encoding="utf-8") as f:
    f.write("\n".join(urls))

print(f"{len(urls)} URLs opgeslagen in '{URLS_BESTAND}'")

Token opgehaald
  Pagina 1: 20/20 URLs verzameld
20 URLs opgeslagen in 'recepten_urls.txt'


---
## Stap 3: Recepten scrapen

Per URL wordt de HTML-pagina gedownload en geparsed met BeautifulSoup. Ingrediënten en stappen staan in vaste HTML-structuren, waardoor een rule-based parser volstaat. De volgende velden worden geëxtraheerd:
- **Naam** — `<h1>`
- **Beschrijving** — eerste `<p>` na `<h1>`
- **Ingrediënten** — `<ul>` onder de `Ingrediënten`-heading
- **Stappen** — `<ol>` onder de `Aan de slag`-heading
- **Benodigdheden** — `<ul>` onder `Dit heb je nodig`

In [6]:
def scrape_recept(url: str) -> dict:
    """Scrapt één recept en geeft een dictionary terug."""
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # Naam
    naam = soup.find("h1").get_text(strip=True) if soup.find("h1") else ""

    # Beschrijving
    beschrijving = ""
    h1 = soup.find("h1")
    if h1:
        sibling = h1.find_next_sibling()
        if sibling and sibling.name == "p":
            beschrijving = sibling.get_text(strip=True)

    # Ingrediënten
    ingredienten = []
    ing_heading = soup.find(["h2", "h3"], string=re.compile(r"Ingredi", re.I))
    if ing_heading:
        ul = ing_heading.find_next("ul")
        if ul:
            for li in ul.find_all("li"):
                tekst = li.get_text(" ", strip=True)
                if tekst:
                    ingredienten.append(tekst)

    # Stappen
    stappen = []
    stap_heading = soup.find(["h2", "h3"], string=re.compile(r"Aan de slag", re.I))
    if stap_heading:
        ol = stap_heading.find_next("ol")
        if ol:
            for li in ol.find_all("li"):
                tekst = re.sub(r"^\d+\s+", "", li.get_text(" ", strip=True)).strip()
                if tekst:
                    stappen.append(tekst)

    # Benodigdheden
    benodigdheden = []
    benodigd_heading = soup.find(["h3", "h4"], string=re.compile(r"Dit heb je nodig", re.I))
    if benodigd_heading:
        ul = benodigd_heading.find_next("ul")
        if ul:
            for li in ul.find_all("li"):
                tekst = li.get_text(strip=True)
                if tekst:
                    benodigdheden.append(tekst)

    # Recept-ID uit URL
    recept_id_match = re.search(r"R-R(\d+)", url)
    recept_id = recept_id_match.group(1) if recept_id_match else ""

    return {
        "ID":            recept_id,
        "Naam":          naam,
        "Beschrijving":  beschrijving,
        "Ingrediënten":  "\n".join(ingredienten),
        "Stappen":       "\n".join(f"{i+1}. {s}" for i, s in enumerate(stappen)),
        "Benodigdheden": "\n".join(benodigdheden),
        "URL":           url,
    }

In [7]:
# Lees URLs en scrape recepten
with open(URLS_BESTAND, "r", encoding="utf-8") as f:
    urls = [r.strip() for r in f.readlines() if r.strip()]

print(f"{len(urls)} URLs gevonden in '{URLS_BESTAND}'\n")

recepten = []
for i, url in enumerate(urls, 1):
    print(f"[{i}/{len(urls)}] {url.split('/')[-1]}", end=" ... ", flush=True)
    try:
        recept = scrape_recept(url)
        recepten.append(recept)
        ing_count  = len([x for x in recept["Ingrediënten"].splitlines() if x])
        stap_count = len([x for x in recept["Stappen"].splitlines() if x])
        print(f"({ing_count} ingrediënten, {stap_count} stappen)")
    except Exception as e:
        print(f"X Fout: {e}")
    time.sleep(DELAY_SECONDEN)

print(f"\n── Samenvatting ───────────────────────────────")
print(f"  Totaal gescraped : {len(recepten)} recepten")
if recepten:
    print(f"  Eerste recept    : {recepten[0]['Naam']}")

20 URLs gevonden in 'recepten_urls.txt'

(12 ingrediënten, 4 stappen)
[2/20]  ... (7 ingrediënten, 2 stappen)
[3/20]  ... X Fout: 404 Client Error: Not Found for url: https://www.ah.nl/allerhande/recept/R-R1202375
[4/20]  ... (14 ingrediënten, 4 stappen)
[5/20]  ... (9 ingrediënten, 3 stappen)
[6/20]  ... (8 ingrediënten, 3 stappen)
[7/20]  ... (8 ingrediënten, 4 stappen)
[8/20]  ... (8 ingrediënten, 3 stappen)
[9/20]  ... (7 ingrediënten, 3 stappen)
[10/20]  ... (5 ingrediënten, 2 stappen)
[11/20]  ... (5 ingrediënten, 1 stappen)
[12/20]  ... (4 ingrediënten, 1 stappen)
[13/20]  ... (4 ingrediënten, 1 stappen)
[14/20]  ... (4 ingrediënten, 1 stappen)
[15/20]  ... (4 ingrediënten, 2 stappen)
[16/20]  ... (4 ingrediënten, 1 stappen)
[17/20]  ... (5 ingrediënten, 1 stappen)
[18/20]  ... (4 ingrediënten, 1 stappen)
[19/20]  ... (4 ingrediënten, 2 stappen)
[20/20]  ... (5 ingrediënten, 2 stappen)

── Samenvatting ───────────────────────────────
  Totaal gescraped : 19 recepten
  Eerste rec

---
## Stap 4: Exporteren naar Excel

Alle recepten worden opgeslagen in een geformatteerd Excel-bestand met tekst-wrapping en aangepaste kolombreedtes.

In [8]:
def exporteer_naar_excel(recepten: list, pad: str) -> None:
    """Exporteert recepten naar een geformatteerd Excel-bestand."""
    df = pd.DataFrame(recepten)

    with pd.ExcelWriter(pad, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Recepten")

        ws = writer.sheets["Recepten"]
        kolom_breedtes = {
            "A": 12, "B": 45, "C": 50,
            "D": 50, "E": 60, "F": 30, "G": 60,
        }
        for kolom, breedte in kolom_breedtes.items():
            ws.column_dimensions[kolom].width = breedte

        for rij in ws.iter_rows(min_row=2):
            for cel in rij:
                cel.alignment = Alignment(wrap_text=True, vertical="top")

    print(f"{len(recepten)} recepten opgeslagen in '{pad}'")


if recepten:
    exporteer_naar_excel(recepten, RECEPTEN_BESTAND)
    df_recepten = pd.DataFrame(recepten)
    display(df_recepten.head(3))
else:
    print("✗ Geen recepten om te exporteren.")

19 recepten opgeslagen in 'recepten.xlsx'


,ID,Naam,Beschrijving,Ingrediënten,Stappen,Benodigdheden,URL
0,1202387,Bloemisalade met whipped geitenkaas,"De bloemi in deze gezonde, glutenvrije salade ...",50 g radijsjes\n30 ml appelciderazijn\n50 ml k...,1. Snijd de radijs in dunne plakken. Doe in ee...,grillpan\nstaafmixer met hoge beker,https://www.ah.nl/allerhande/recept/R-R1202387/
1,1202381,Eiersalade met tuinerwten,Vervang de mayonaise in je eiersalade 'ns door...,200 g diepvries tuinerwten\n12 middelgrote sch...,1. Laat de tuinerwten ontdooien. Kook de eiere...,,https://www.ah.nl/allerhande/recept/R-R1202381/
2,1202372,Gezond paaseibroodje met spinazie,Dit gezonde paaseibroodje met spinazie is een ...,3.5 g gedroogde gist\n125 ml lauwwarm water\n5...,1. Los de gist op in 50 ml lauwwarm water en l...,staande mixer met deeghaken\nbakpapier,https://www.ah.nl/allerhande/recept/R-R1202372/


---
## Stap 5: Pantry-proxies genereren

Omdat echte gebruikerspantry-data moeilijk te verzamelen is, worden synthetische pantry's aangemaakt voor **100 gebruikers**.

Elke gebruiker krijgt een profiel toegewezen:

| Profiel | Gewicht | Kans-factor |
|---|---|---|
| `gezin` | 25% | 1.2 (meer ingrediënten) |
| `student` | 20% | 0.7 (minder ingrediënten) |
| `stel` | 25% | 1.0 (gemiddeld) |
| `alleenstaand` | 20% | 0.8 |
| `gezond_eter` | 10% | 1.1 |

In [9]:
# ── Meervoud → enkelvoud normalisatie ─────────────────────────────────────────

def enkelvoud(s: str) -> str:
    """Zet gangbare Nederlandse meervoudsvormen om naar enkelvoud."""
    woorden = s.split()
    if not woorden:
        return s
    last = woorden[-1]
    uitzonderingen = {
        'eieren': 'ei', 'uien': 'ui', 'appels': 'appel', 'appelen': 'appel',
        'peren': 'peer', 'citroenen': 'citroen', 'tomaten': 'tomaat',
        'aardappelen': 'aardappel', 'wortelen': 'wortel', 'paprikas': 'paprika',
        'champignons': 'champignon', 'courgettes': 'courgette',
        'kipfilets': 'kipfilet', 'kipdijfilets': 'kipdijfilet',
        'biefstukken': 'biefstuk', 'rookworsten': 'rookworst',
        'scharreleieren': 'scharrelei', 'bosuitjes': 'bosui',
        'krieltjes': 'krieltje', 'blokjes': 'blokje',
    }
    low = last.lower()
    if low in uitzonderingen:
        woorden[-1] = uitzonderingen[low]
    elif low.endswith('jes') and len(low) > 5:
        woorden[-1] = last[:-2]
    elif low.endswith('tjes') and len(low) > 6:
        woorden[-1] = last[:-3] + 'je'
    return ' '.join(woorden)


# ── Pantry-definitie ──────────────────────────────────────────────────────────
# Formaat: (naam, eenheid, min_hoeveelheid, max_hoeveelheid, kans_aanwezig)

PANTRY = {
    'zuivel': [
        ('ei', 'stuk', 2, 12, 0.90), ('scharrelei', 'stuk', 2, 12, 0.60),
        ('melk', 'ml', 250, 1500, 0.85), ('halfvolle melk', 'ml', 250, 1000, 0.70),
        ('volle melk', 'ml', 250, 1000, 0.40), ('boter', 'g', 50, 250, 0.85),
        ('roomboter', 'g', 50, 250, 0.70), ('Griekse yoghurt', 'g', 150, 500, 0.55),
        ('volle yoghurt', 'g', 150, 500, 0.50), ('crème fraîche', 'g', 100, 200, 0.45),
        ('slagroom', 'ml', 100, 250, 0.40), ('kookroom', 'ml', 100, 200, 0.35),
        ('geraspte kaas', 'g', 50, 200, 0.70), ('Parmezaanse kaas', 'g', 30, 150, 0.50),
        ('mozzarella', 'g', 100, 250, 0.45), ('feta', 'g', 100, 200, 0.40),
        ('Griekse feta', 'g', 100, 200, 0.35), ('ricotta', 'g', 125, 250, 0.30),
        ('cottagecheese', 'g', 150, 300, 0.30), ('roomkaas', 'g', 100, 200, 0.40),
        ('mascarpone', 'g', 100, 250, 0.30), ('kwark', 'g', 150, 500, 0.45),
    ],
    'groente_vers': [
        ('ui', 'stuk', 1, 5, 0.95), ('rode ui', 'stuk', 1, 3, 0.60),
        ('knoflook', 'stuk', 1, 3, 0.90), ('tomaat', 'stuk', 1, 6, 0.75),
        ('paprika', 'stuk', 1, 3, 0.65), ('rode paprika', 'stuk', 1, 3, 0.55),
        ('wortel', 'stuk', 1, 5, 0.70), ('aardappel', 'g', 200, 1000, 0.80),
        ('courgette', 'stuk', 1, 2, 0.50), ('champignon', 'g', 100, 400, 0.55),
        ('kastanjechampignon', 'g', 100, 250, 0.40), ('spinazie', 'g', 100, 400, 0.50),
        ('sla', 'stuk', 1, 2, 0.45), ('rucola', 'g', 50, 150, 0.40),
        ('prei', 'stuk', 1, 2, 0.45), ('broccoli', 'stuk', 1, 2, 0.50),
        ('bloemkool', 'stuk', 1, 1, 0.40), ('citroen', 'stuk', 1, 4, 0.70),
        ('limoen', 'stuk', 1, 3, 0.45), ('verse gember', 'g', 20, 100, 0.50),
        ('cherrytomaat', 'g', 150, 400, 0.50), ('avocado', 'stuk', 1, 3, 0.50),
        ('bosui', 'stuk', 2, 6, 0.45), ('venkel', 'stuk', 1, 2, 0.30),
        ('zoete aardappel', 'g', 200, 600, 0.40), ('pompoen', 'g', 300, 800, 0.30),
        ('asperge', 'g', 200, 500, 0.30), ('sperzieboon', 'g', 150, 400, 0.35),
        ('ijsbergsla', 'stuk', 1, 1, 0.40), ('komkommer', 'stuk', 1, 2, 0.50),
    ],
    'vlees_vis': [
        ('kipfilet', 'g', 200, 600, 0.70), ('kipdijfilet', 'g', 200, 500, 0.55),
        ('kipfiletreepje', 'g', 200, 400, 0.45), ('gehakt', 'g', 250, 500, 0.65),
        ('rundergehakt', 'g', 250, 500, 0.55), ('half-om-halfgehakt', 'g', 250, 500, 0.50),
        ('zalm', 'g', 150, 400, 0.55), ('gerookte zalm', 'g', 100, 200, 0.45),
        ('tonijn in blik', 'g', 100, 200, 0.60), ('garnaal', 'g', 150, 300, 0.35),
        ('bacon', 'g', 75, 200, 0.55), ('spekreepje', 'g', 75, 150, 0.45),
        ('kipgehakt', 'g', 250, 500, 0.35), ('biefstuk', 'stuk', 1, 2, 0.35),
        ('gehaktbal', 'stuk', 4, 12, 0.40), ('rookworst', 'stuk', 1, 2, 0.40),
    ],
    'droogwaren': [
        ('pasta', 'g', 250, 500, 0.85), ('spaghetti', 'g', 250, 500, 0.75),
        ('penne', 'g', 250, 500, 0.60), ('fusilli', 'g', 250, 500, 0.55),
        ('rijst', 'g', 250, 1000, 0.85), ('basmatirijst', 'g', 250, 750, 0.60),
        ('bloem', 'g', 100, 1000, 0.75), ('zelfrijzend bakmeel', 'g', 100, 500, 0.45),
        ('paneermeel', 'g', 100, 300, 0.40), ('havermout', 'g', 100, 500, 0.55),
        ('linze', 'g', 150, 400, 0.45), ('kikkererwt', 'g', 150, 400, 0.45),
        ('kidneyboon', 'g', 150, 400, 0.40), ('bruine boon', 'g', 150, 400, 0.35),
        ('couscous', 'g', 150, 400, 0.45), ('quinoa', 'g', 150, 300, 0.40),
        ('lasagneblad', 'stuk', 6, 12, 0.35), ('crackers', 'stuk', 4, 16, 0.50),
        ('broodkruimel', 'g', 50, 200, 0.35),
    ],
    'sauzen_conserven': [
        ('tomatensaus', 'g', 200, 400, 0.70), ('passata', 'g', 200, 700, 0.60),
        ('tomatenblokje in blik', 'g', 200, 400, 0.65), ('tomatenconcentraat', 'g', 70, 140, 0.60),
        ('kokosmelk', 'ml', 200, 400, 0.55), ('bouillonblokje', 'stuk', 2, 6, 0.80),
        ('kippenbouillonblokje', 'stuk', 2, 4, 0.70), ('groentebouillonblokje', 'stuk', 2, 4, 0.65),
        ('pesto', 'g', 80, 190, 0.55), ('sojasaus', 'ml', 50, 200, 0.65),
        ('ketjap manis', 'ml', 50, 150, 0.50), ('Worcestershiresaus', 'ml', 30, 100, 0.40),
        ('mayonaise', 'g', 100, 400, 0.70), ('mosterd', 'g', 100, 300, 0.75),
        ('ketchup', 'g', 100, 400, 0.70), ('sambal', 'g', 50, 200, 0.55),
        ('sriracha', 'ml', 50, 150, 0.40), ('appelciderazijn', 'ml', 50, 250, 0.55),
        ('rode wijnazijn', 'ml', 50, 250, 0.50), ('balsamicoazijn', 'ml', 50, 150, 0.55),
        ('honing', 'g', 100, 400, 0.70), ('ahornsiroop', 'ml', 50, 250, 0.35),
        ('pindasaus', 'g', 100, 300, 0.45), ('tahini', 'g', 100, 300, 0.40),
        ('pindakaas', 'g', 100, 400, 0.65),
    ],
    'olie_vet': [
        ('olijfolie', 'ml', 100, 750, 0.90), ('zonnebloemolie', 'ml', 100, 500, 0.70),
        ('olie', 'ml', 100, 500, 0.60), ('sesamolie', 'ml', 30, 100, 0.40),
    ],
    'kruiden_specerijen': [
        ('zout', 'g', 50, 500, 0.99), ('zwarte peper', 'g', 20, 100, 0.99),
        ('paprikapoeder', 'g', 20, 80, 0.75), ('komijn', 'g', 10, 50, 0.60),
        ('koriander', 'g', 10, 50, 0.55), ('kurkuma', 'g', 10, 50, 0.55),
        ('kaneel', 'g', 10, 50, 0.65), ('nootmuskaat', 'g', 5, 30, 0.60),
        ('oregano', 'g', 5, 30, 0.70), ('tijm', 'g', 5, 30, 0.65),
        ('rozemarijn', 'g', 5, 30, 0.55), ('basilicum', 'g', 5, 30, 0.65),
        ('knoflookpoeder', 'g', 10, 50, 0.60), ('cayennepeper', 'g', 5, 30, 0.55),
        ('chilipoeder', 'g', 5, 30, 0.50), ('kerriepoeder', 'g', 10, 50, 0.55),
        ('garam masala', 'g', 10, 40, 0.40), ('Italiaanse kruiden', 'g', 10, 40, 0.65),
        ('laurierblad', 'stuk', 2, 8, 0.60), ('verse basilicum', 'g', 15, 30, 0.45),
        ('verse peterselie', 'g', 15, 30, 0.50), ('verse tijm', 'g', 10, 20, 0.35),
        ('verse koriander', 'g', 15, 30, 0.40), ('verse munt', 'g', 10, 20, 0.35),
        ('bieslook', 'g', 10, 20, 0.40),
    ],
    'bakken_zoet': [
        ('suiker', 'g', 100, 1000, 0.80), ('basterdsuiker', 'g', 100, 500, 0.55),
        ('poedersuiker', 'g', 50, 300, 0.45), ('bruine suiker', 'g', 100, 500, 0.50),
        ('vanillesuiker', 'g', 20, 80, 0.60), ('bakpoeder', 'g', 10, 100, 0.65),
        ('baksoda', 'g', 10, 100, 0.45), ('cacao', 'g', 50, 200, 0.50),
        ('pure chocolade', 'g', 50, 200, 0.45), ('melkchocolade', 'g', 50, 200, 0.40),
        ('rozijn', 'g', 50, 200, 0.45), ('amandel', 'g', 50, 200, 0.45),
        ('walnoot', 'g', 50, 200, 0.40), ('cashewnoot', 'g', 50, 200, 0.40),
        ('pijnboompit', 'g', 30, 100, 0.40), ('gedroogde abrikoos', 'g', 50, 200, 0.30),
        ('kokosrasp', 'g', 50, 150, 0.35), ('maizena', 'g', 50, 200, 0.60),
        ('gelatineblad', 'stuk', 2, 8, 0.30), ('vanilleextract', 'ml', 10, 50, 0.40),
    ],
    'brood_ontbijt': [
        ('brood', 'stuk', 1, 2, 0.80), ('volkoren brood', 'stuk', 1, 1, 0.55),
        ('wrap', 'stuk', 2, 8, 0.55), ('tortilla', 'stuk', 2, 8, 0.50),
        ('beschuit', 'stuk', 3, 12, 0.55), ('roggebrood', 'stuk', 1, 1, 0.35),
        ('müsli', 'g', 150, 500, 0.45), ('granola', 'g', 100, 400, 0.40),
        ('cornflakes', 'g', 100, 400, 0.40), ('jam', 'g', 100, 400, 0.65),
        ('appelstroop', 'g', 100, 400, 0.40), ('hagelslag', 'g', 100, 400, 0.50),
        ('pindakaas', 'g', 100, 400, 0.65),
    ],
    'divers': [
        ('witte wijn', 'ml', 150, 750, 0.45), ('rode wijn', 'ml', 150, 750, 0.45),
        ('bier', 'ml', 330, 660, 0.40), ('kippenboullion', 'ml', 200, 1000, 0.40),
        ('groentegroenbouillon', 'ml', 200, 1000, 0.35),
    ]
}

print(f"Pantry-definitie geladen: {sum(len(v) for v in PANTRY.values())} ingrediënten in {len(PANTRY)} categorieën")

Pantry-definitie geladen: 179 ingrediënten in 10 categorieën


In [10]:
# ── Gebruikersprofielen & kans-factoren ───────────────────────────────────────

user_profiles    = ['gezin', 'student', 'stel', 'alleenstaand', 'gezond_eter']
profile_weights  = [0.25, 0.20, 0.25, 0.20, 0.10]
kans_factoren    = {'gezin': 1.2, 'student': 0.7, 'stel': 1.0, 'alleenstaand': 0.8, 'gezond_eter': 1.1}

# Platte opzoektabel van ingrediënten
pantry_items = {}
for cat, items in PANTRY.items():
    for naam, eenheid, mn, mx, kans in items:
        pantry_items[naam] = (eenheid, kans, mn, mx, cat)

# ── Genereer pantry's voor 100 gebruikers ─────────────────────────────────────
rows = []

for user_id in range(1, 101):
    profiel     = random.choices(user_profiles, weights=profile_weights)[0]
    kans_factor = kans_factoren[profiel]

    for ingredient, (eenheid, kans, mn, mx, categorie) in pantry_items.items():
        aangepaste_kans = min(kans * kans_factor, 0.99)
        if random.random() < aangepaste_kans:
            if eenheid in ('g', 'ml'):
                stap = 50 if mx > 200 else 25
                hoeveelheid = round(random.randint(mn, mx) / stap) * stap
                hoeveelheid = max(mn, hoeveelheid)
            else:
                hoeveelheid = random.randint(mn, mx)

            rows.append({
                'user_id':    user_id,
                'profiel':    profiel,
                'categorie':  categorie,
                'ingredient': ingredient,
                'hoeveelheid': hoeveelheid,
                'eenheid':    eenheid,
            })

df_pantry = pd.DataFrame(rows)

print(f"Pantry-proxies gegenereerd")
print(f"  Totaal rijen              : {len(df_pantry)}")
print(f"  Gem. ingrediënten per user: {len(df_pantry) / 100:.1f}")
print("\nVerdeling over profielen:")
print(df_pantry.groupby('profiel')['user_id'].nunique().to_string())
print("\nVoorbeeld (eerste 5 rijen):")
display(df_pantry.head())

Pantry-proxies gegenereerd
  Totaal rijen              : 9016
  Gem. ingrediënten per user: 90.2

Verdeling over profielen:
profiel
alleenstaand    15
gezin           27
gezond_eter     11
stel            21
student         26

Voorbeeld (eerste 5 rijen):


,user_id,profiel,categorie,ingredient,hoeveelheid,eenheid
0,1,stel,zuivel,ei,6,stuk
1,1,stel,zuivel,scharrelei,4,stuk
2,1,stel,zuivel,melk,1350,ml
3,1,stel,zuivel,halfvolle melk,700,ml
4,1,stel,zuivel,volle melk,350,ml


---
## Stap 6: Pantry-data exporteren naar Excel

Het Excel-bestand bevat drie tabbladen:
- **Pantry Data** — alle rijen met kleurcodering per categorie
- **Samenvatting per User** — aantal ingrediënten en categorieën per gebruiker
- **Ingredientfrequentie** — hoe vaak elk ingrediënt voorkomt over alle gebruikers

In [11]:
# Sla eerst op als CSV (handig als tussenbestand)
df_pantry.to_csv('pantry_proxies_test.csv', index=False)
print("pantry_proxies_test.csv opgeslagen")

# ── Stijl-constanten ──────────────────────────────────────────────────────────
HEADER_FILL    = PatternFill('solid', start_color='2E7D32')
HEADER_FONT    = Font(name='Arial', bold=True, color='FFFFFF', size=11)
RAND_STIJL     = Border(
    bottom=Side(style='thin', color='C8E6C9'),
    right=Side(style='thin', color='C8E6C9'),
)
CAT_COLORS = {
    'zuivel':             'E3F2FD', 'groente_vers':       'E8F5E9',
    'vlees_vis':          'FCE4EC', 'droogwaren':         'FFF8E1',
    'sauzen_conserven':   'FFF3E0', 'olie_vet':           'F3E5F5',
    'kruiden_specerijen': 'E0F7FA', 'bakken_zoet':        'FBE9E7',
    'brood_ontbijt':      'EFEBE9', 'divers':             'FAFAFA',
}
PROFIEL_KLEUREN = {
    'gezin': 'BBDEFB', 'student': 'C8E6C9', 'stel': 'FFE0B2',
    'alleenstaand': 'F8BBD0', 'gezond_eter': 'E1BEE7',
}

def maak_header_rij(ws, headers, fill, font):
    """Voeg een gestijlde header-rij toe aan een werkblad."""
    ws.append(headers)
    ws.row_dimensions[1].height = 22
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = font
        cell.fill = fill
        cell.alignment = Alignment(horizontal='center', vertical='center')

wb = Workbook()

# ── Tabblad 1: Pantry Data ────────────────────────────────────────────────────
ws1 = wb.active
ws1.title = 'Pantry Data'
headers1 = ['user_id', 'profiel', 'categorie', 'ingredient', 'hoeveelheid', 'eenheid']
maak_header_rij(ws1, headers1, HEADER_FILL, HEADER_FONT)

for row_idx, row in df_pantry.iterrows():
    ws1.append(row.tolist())
    excel_row = row_idx + 2
    fill = PatternFill('solid', start_color=CAT_COLORS.get(row['categorie'], 'FFFFFF'))
    for col_idx in range(1, len(headers1) + 1):
        cell = ws1.cell(row=excel_row, column=col_idx)
        cell.fill = fill
        cell.font = Font(name='Arial', size=10)
        cell.border = RAND_STIJL
        if col_idx in (1, 5):
            cell.alignment = Alignment(horizontal='center')

for i, w in enumerate([10, 15, 20, 35, 14, 10], 1):
    ws1.column_dimensions[get_column_letter(i)].width = w
ws1.freeze_panes = 'A2'
ws1.auto_filter.ref = f"A1:F{len(df_pantry)+1}"

# ── Tabblad 2: Samenvatting per User ─────────────────────────────────────────
ws2 = wb.create_sheet('Samenvatting per User')
summary = df_pantry.groupby(['user_id', 'profiel']).agg(
    aantal_ingredienten=('ingredient', 'count'),
    categorieen=('categorie', 'nunique'),
).reset_index()

headers2 = ['user_id', 'profiel', 'aantal_ingredienten', 'categorieen']
maak_header_rij(ws2, headers2, HEADER_FILL, HEADER_FONT)

for row_idx, row in summary.iterrows():
    ws2.append(row.tolist())
    excel_row = row_idx + 2
    fill = PatternFill('solid', start_color=PROFIEL_KLEUREN.get(row['profiel'], 'FFFFFF'))
    for col_idx in range(1, 5):
        cell = ws2.cell(row=excel_row, column=col_idx)
        cell.fill = fill
        cell.font = Font(name='Arial', size=10)
        cell.border = RAND_STIJL
        cell.alignment = Alignment(horizontal='center')

for i, w in enumerate([10, 15, 22, 15], 1):
    ws2.column_dimensions[get_column_letter(i)].width = w
ws2.freeze_panes = 'A2'

# ── Tabblad 3: Ingredientfrequentie ──────────────────────────────────────────
ws3 = wb.create_sheet('Ingredientfrequentie')
freq = df_pantry.groupby(['ingredient', 'categorie'])['user_id'].nunique().reset_index()
freq.columns = ['ingredient', 'categorie', 'aantal_users']
freq = freq.sort_values('aantal_users', ascending=False)

headers3 = ['ingredient', 'categorie', 'aantal_users', '% users']
maak_header_rij(ws3, headers3, HEADER_FILL, HEADER_FONT)

for row_idx, row in freq.iterrows():
    pct = f"=C{row_idx+2}/100"
    ws3.append([row['ingredient'], row['categorie'], row['aantal_users'], pct])
    excel_row = row_idx + 2
    fill = PatternFill('solid', start_color=CAT_COLORS.get(row['categorie'], 'FFFFFF'))
    for col_idx in range(1, 5):
        cell = ws3.cell(row=excel_row, column=col_idx)
        cell.fill = fill
        cell.font = Font(name='Arial', size=10)
        cell.border = RAND_STIJL
        if col_idx == 3:
            cell.alignment = Alignment(horizontal='center')
        if col_idx == 4:
            cell.number_format = '0.0%'

for i, w in enumerate([35, 20, 14, 12], 1):
    ws3.column_dimensions[get_column_letter(i)].width = w
ws3.freeze_panes = 'A2'
ws3.auto_filter.ref = f"A1:D{len(freq)+1}"

wb.save('pantry_proxies_test.xlsx')
print("\npantry_proxies_test.xlsx opgeslagen met 3 tabbladen:")
print("   - Pantry Data")
print("   - Samenvatting per User")
print("   - Ingredientfrequentie")

pantry_proxies_test.csv opgeslagen

pantry_proxies_test.xlsx opgeslagen met 3 tabbladen:
   - Pantry Data
   - Samenvatting per User
   - Ingredientfrequentie


---
## Klaar!

De volgende bestanden zijn aangemaakt:

| Bestand | Beschrijving |
|---|---|
| `recepten_urls.txt` | Lijst met alle recept-URLs |
| `recepten.xlsx` | Gescrapete recepten met ingrediënten en stappen |
| `pantry_proxies_test.csv` | Pantry-data als CSV |
| `pantry_proxies_test.xlsx` | Pantry-data als geformatteerd Excel-bestand |

In [12]:
import os

bestanden = ['recepten_urls.txt', 'recepten.xlsx', 'pantry_proxies_test.csv', 'pantry_proxies_test.xlsx']
print("── Gegenereerde bestanden ──────────────────────")
for b in bestanden:
    if os.path.exists(b):
        grootte = os.path.getsize(b)
        print(f"  {b:<30} ({grootte:,} bytes)")
    else:
        print(f"  ✗ {b} — niet gevonden")

── Gegenereerde bestanden ──────────────────────
  recepten_urls.txt              (978 bytes)
  recepten.xlsx                  (11,391 bytes)
  pantry_proxies_test.csv        (372,901 bytes)
  pantry_proxies_test.xlsx       (290,446 bytes)
